# Customers Segmentation

In [1]:
import gc
gc.collect()

35

In [4]:
import numpy as np
import pandas as pd
# pd.set_option('max_columns', 150)

import gc
import os

# matplotlib and seaborn for plotting
import matplotlib
# matplotlib.rcParams['figure.dpi'] = 120 #resolution
# matplotlib.rcParams['figure.figsize'] = (8,6) #figure size

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')
color = sns.color_palette()

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.decomposition import PCA

root = r"C:\Users\MHMD RAGAB\Downloads\DATA\New folder\\"

## Data

In [5]:
aisles = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\aisles.csv')
departments = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\departments.csv')
order_products__prior = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\order_products__prior.csv')
order_products__train = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\order_products__train.csv')
orders = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\orders.csv')
products = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\products.csv')

For segmentation I am considering users from prior set only

In [7]:
order_products = order_products__prior.merge(products, on ='product_id', how='left')
order_products = order_products.merge(aisles, on ='aisle_id', how='left')
order_products = order_products.merge(departments, on ='department_id', how='left')
order_products = order_products.merge(orders, on='order_id', how='left')
order_products.shape

(32434489, 15)

In [8]:
order_products.head()

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2,33120,1,1,Organic Egg Whites,86,16,eggs,dairy eggs,202279,prior,3,5,9,8.0
1,2,28985,2,1,Michigan Organic Kale,83,4,fresh vegetables,produce,202279,prior,3,5,9,8.0
2,2,9327,3,0,Garlic Powder,104,13,spices seasonings,pantry,202279,prior,3,5,9,8.0
3,2,45918,4,1,Coconut Butter,19,13,oils vinegars,pantry,202279,prior,3,5,9,8.0
4,2,30035,5,0,Natural Sweetener,17,13,baking ingredients,pantry,202279,prior,3,5,9,8.0


In [9]:
order_products.user_id.nunique()

206209

## Segmentation

Since there are thousands of products in the dataset I will rely on aisles, which represent categories of products. Even with aisles features will be too much so I will use Principal Component Analysis to find new dimensions along which clustering will be easier.

In [ ]:
cross_df = pd.crosstab(order_products.user_id, order_products.aisle)
cross_df.head()

In [ ]:
cross_df.tail()

I will normalize each row

In [ ]:
df = cross_df.div(cross_df.sum(axis=1), axis=0)
df.head()

In [ ]:
df.shape

### PCA and K-Means Clustering

Reducing this dataframe to only 10 dimensions as KMeans does not work properly in higher dimension. 

In [ ]:
pca = PCA(n_components=10)
df_pca = pca.fit_transform(df)
df_pca = pd.DataFrame(df_pca)
df_pca.head()

In [ ]:
Sum_of_squared_distances = []
K = range(1,10)
for k in K:
    km = KMeans(n_clusters=k)
    km = km.fit(df_pca)
    Sum_of_squared_distances.append(km.inertia_)

In [ ]:
plt.subplots(figsize = (8, 5))
plt.plot(K, Sum_of_squared_distances, 'bx-')
plt.xlabel('k')
plt.ylabel('Sum_of_squared_distances')
plt.title('Elbow Method For Optimal k')
plt.show()

From above plot we can choose optimal K as 5

In [ ]:
clusterer = KMeans(n_clusters=5,random_state=42).fit(df_pca)
centers = clusterer.cluster_centers_
c_preds = clusterer.predict(df_pca)
print(centers)

#### Visualizing clustering among first two principal components

In [ ]:
temp_df = df_pca.iloc[:, 0:2]
temp_df.columns = ["pc1", "pc2"]
temp_df['cluster'] = c_preds
temp_df.head()

In [ ]:
fig, ax = plt.subplots(figsize = (8, 5))
ax = sns.scatterplot(data = temp_df, x = "pc1", y = "pc2", hue = "cluster")
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
ax.set_title("Cluster Visualization")
plt.show();

### Top products per cluster

In [ ]:
cross_df['cluster'] = c_preds

cluster1 = cross_df[cross_df.cluster == 0]
cluster2 = cross_df[cross_df.cluster == 1]
cluster3 = cross_df[cross_df.cluster == 2]
cluster4 = cross_df[cross_df.cluster == 3]
cluster5 = cross_df[cross_df.cluster == 4]

In [ ]:
cluster1.shape

In [ ]:
cluster1.drop('cluster',axis=1).mean().sort_values(ascending=False)[0:10]

In [ ]:
cluster2.shape

In [ ]:
cluster2.drop('cluster',axis=1).mean().sort_values(ascending=False)[0:10]

In [ ]:
cluster3.shape

In [ ]:
cluster3.drop('cluster',axis=1).mean().sort_values(ascending=False)[0:10]

In [ ]:
cluster4.shape

In [ ]:
cluster4.drop('cluster',axis=1).mean().sort_values(ascending=False)[0:10]

In [ ]:
cluster5.shape

In [ ]:
cluster5.drop('cluster',axis=1).mean().sort_values(ascending=False)[0:10]

Customer Segmentation Results:

- Cluster 1 results into 5428 consumers having a very strong preference for water seltzer sparkling water aisle.
- Cluster 2 results into 55784 consumers who mostly order fresh vegetables followed by fruits.
- Cluster 3 results into 7948 consumers who buy packaged produce and fresh fruits mostly.
- Cluster 4 results into 37949 consumers who have a very strong preference for fruits followed by fresh vegetables.
- Cluster 5 results into 99100 consumers who orders products from many aisles. Their mean orders are low compared to other clusters which tells us that either they are not frequent users of Instacart or they are new users and do not have many orders yet. 